In [7]:
import base64
import json
import requests

In [ ]:
headers = {
    "Authorization": "Bearer ***"
}

In [4]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 2

In [ ]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATION_IDS = [org["id"] for org in response.json()["data"]]
ORGANIZATION_IDS

[1, 3]

In [11]:
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
SESSION_ID = 3

In [20]:
# When a new cohort is created, vantage6 needs to extract the data from the OMOP
# database and store it in the session as a dataframe. This is done by executing a
# vantage6 extraction task.
#
# This cell contains all the parameters that are going to end up in the `payload`
# dictionary.
#

#
# Static content
#
image = "harbor2.vantage6.ai/idea4rc/sessions:latest"
label = "omop"

#
# Dynamic content
#
# The name of the cohort, this should be unique within a session. You can probably use
# the same name that you use in the RAVEN UI. Alternatively, we can also not send it.
# In that case the name will be generated by vantage6.
# name = "Cohort_name_84"

# Each `image` can have multiple `methods`. To extract the data from the OMOP database
# we need to use the `create_cohort` method.
method = "create_cohort"

# The input for the task is the patient ids and which features we want to extract.
arguments = {
    # NOTE --- CHANGE THE PATIENT IDS TO THE PATIENT IDS OF THE COHORT ---
    # These `patient_ids` should be coming from the cohort builder in RAVEN
    "patient_ids": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    # NOTE --- CHANGE THE FEATURES TO THE FEATURES OF THE COHORT ---
    # The features are the features that we want to extract from the OMOP database.
    # This can be either "sarcoma" or "head_neck".
    # TODO The "head_neck is not implemented yet.
    "features": "sarcoma"
}

In [21]:
# before we can create a task we need to prepare task instructions. In vantage6 we can
# (but we dont in IDEA4RC) use end-to-end encryption, therefore we need to store the
# input for each organization individually.
payload = {
    "label": label,
    # "name": name, # optional, v6 will generate a name if not provided
    "task": {
        "method": method,
        "image": image,
        # In vantage6 we can (but we dont in IDEA4RC) use end-to-end encryption,
        # therefore we need to store the input for each organization individually.
        "organizations": [
            {
                "id": id_,
                "arguments": base64.b64encode(
                    json.dumps(arguments).encode("UTF-8")
                ).decode("UTF-8")
            }
            # We always create a cohort for all organizations in the study. Even though
            # in a later stage we might send computation tasks to a subset of the
            # organizations.
            for id_ in ORGANIZATION_IDS
        ]
    }
}
payload

{'label': 'omop',
 'task': {'method': 'create_cohort',
  'image': 'harbor2.vantage6.ai/idea4rc/sessions:latest',
  'organizations': [{'id': 1,
    'arguments': 'eyJwYXRpZW50X2lkcyI6IFsxLCAyLCAzLCA0LCA1LCA2LCA3LCA4LCA5LCAxMF0sICJmZWF0dXJlcyI6ICJzYXJjb21hIn0='},
   {'id': 3,
    'arguments': 'eyJwYXRpZW50X2lkcyI6IFsxLCAyLCAzLCA0LCA1LCA2LCA3LCA4LCA5LCAxMF0sICJmZWF0dXJlcyI6ICJzYXJjb21hIn0='}]}}

In [22]:
# Create a vantage6 task to extract the data from the OMOP data source and store it
# into a dataframe.
response = requests.post(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/{SESSION_ID}/dataframe",
    headers=headers,
    json=payload
)
TASK_ID = response.json()["last_session_task"]["id"]
DATAFRAME_ID = response.json()["id"]
response.json()

{'name': 'infallible_hamilton',
 'session': {'id': 3,
  'link': '/server/session/3',
  'methods': ['PATCH', 'DELETE', 'GET']},
 'columns': [],
 'last_session_task': {'study': {'id': 4,
   'link': '/server/study/4',
   'methods': ['PATCH', 'DELETE', 'GET']},
  'finished_at': None,
  'databases': [{'label': 'omop',
    'type': 'source',
    'dataframe_id': None,
    'dataframe_name': None,
    'position': 0}],
  'id': 326,
  'results': '/server/result?task_id=326',
  'parent': None,
  'method': 'create_cohort',
  'session': {'id': 3,
   'link': '/server/session/3',
   'methods': ['PATCH', 'DELETE', 'GET']},
  'created_at': '2025-12-10T12:22:42.496050',
  'required_by': [],
  'children': '/server/task?parent_id=326',
  'collaboration': {'id': 2,
   'link': '/server/collaboration/2',
   'methods': ['PATCH', 'DELETE', 'GET']},
  'status': 'awaiting',
  'name': 'Session initialization (UPM Test Session 0)',
  'dataframe': {'name': 'infallible_hamilton', 'db_label': 'omop', 'id': 88},
  'algo

In [ ]:

# The status of the task (in this case the task that extract the data from the OMOP db
# in order to create the dataframe) can be one of the following:
#
# - pending: The task is waiting to be executed.
# - active: The task is being executed.
# - completed: The task has finished successfully.
# - crashed: The task crashed. You probably want to inspect the logs.
#
# You should poll the status of the task until it got one of the final states: crashed
# or completed
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
response.json()
# The output also shows the status of the nodes, which can be useful to show in the
# RAVEN UI.

{'data': [{'arguments': 'eyJwYXRpZW50X2lkcyI6IFsxLCAyLCAzLCA0LCA1LCA2LCA3LCA4LCA5LCAxMF0sICJmZWF0dXJlcyI6ICJzYXJjb21hIn0=',
   'action': 'data_extraction',
   'cleanup_at': None,
   'finished_at': None,
   'node': {'keycloak_client_id': '089bd4cc-9193-4ba3-970e-ad8e2057b268',
    'name': 'Bilbao---Demo-INT-UPM-node',
    'keycloak_id': 'ff862e61-c20f-487f-9282-93370ecf5dd2',
    'id': 8,
    'status': 'offline'},
   'started_at': None,
   'log': None,
   'task': {'id': 326,
    'link': '/server/task/326',
    'methods': ['GET', 'DELETE']},
   'blob_storage_used': False,
   'id': 427,
   'assigned_at': '2025-12-09T10:18:53.984434',
   'results': {'id': 427,
    'link': '/server/result/427',
    'methods': ['GET', 'PATCH']},
   'organization': {'id': 3,
    'link': '/server/organization/3',
    'methods': ['PATCH', 'DELETE', 'GET']},
   'status': 'pending'},
  {'arguments': 'eyJwYXRpZW50X2lkcyI6IFsxLCAyLCAzLCA0LCA1LCA2LCA3LCA4LCA5LCAxMF0sICJmZWF0dXJlcyI6ICJzYXJjb21hIn0=',
   'action': 'd

In [24]:
# This session ID should be stored in the RAVEN database in the ?? table
# (`v6_dataframe`).
DATAFRAME_ID

88